# Hybrid Search + GraphRAG for Agentic Systems

> Based on: [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)  
> GraphRAG: [Microsoft Research GraphRAG](https://microsoft.github.io/graphrag/)

---

## What We'll Cover

1. **Why pure vector search fails** — the vocabulary gap problem  
2. **BM25 sparse retrieval** — keyword matching with TF-IDF+ scoring  
3. **kNN dense retrieval** — semantic embeddings and HNSW  
4. **Hybrid fusion** — Reciprocal Rank Fusion (RRF) and weighted convex combination  
5. **Agentic RAG loop** — turning retrieval into an LLM-callable tool  
6. **Evaluation** — Hit Rate, MRR, NDCG@K  
7. **GraphRAG** — knowledge graph construction, community detection, local & global search  
8. **Decision guide** — when to use which strategy  

---

### The Core Problem

A standard RAG pipeline looks like this:

```
User query → [Vector DB: kNN search] → Top-K chunks → LLM → Answer
```

This breaks when a user asks:
- `"Error code ER_NO_DEFAULT_FOR_FIELD"` — exact error string (vector search misses it)
- `"What does SKU-4821 cost?"` — product identifier
- `"How are BM25 and HNSW related?"` — multi-hop relationship across documents

The three techniques in this notebook each solve a different failure mode.

## 1. Setup & Dependencies

In [ ]:
%pip install -q rank_bm25 sentence-transformers faiss-cpu anthropic numpy scikit-learn python-dotenv --break-system-packages


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import json
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# BM25
from rank_bm25 import BM25Okapi

# Dense embeddings + FAISS ANN index
from sentence_transformers import SentenceTransformer
import faiss

print("All imports OK")

/Users/rony.boter/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


---
## 2. BM25 — Sparse Retrieval

### Theory

BM25 (*Best Match 25*) is the industry-standard keyword ranking algorithm. It scores every document $d$ against a query $q$ as:

$$\text{score}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

| Symbol | Meaning |
|--------|---------|
| $f(t,d)$ | Term frequency of token $t$ in document $d$ |
| $\|d\|$ | Document length in tokens |
| avgdl | Average document length across corpus |
| $k_1$ | Term saturation (typically 1.2–2.0) |
| $b$ | Length normalization (typically 0.75) |
| IDF | $\log\frac{N - n_t + 0.5}{n_t + 0.5}$ — rewards rare terms |

**Key insight:** BM25 uses an **inverted index** (token → doc list). No GPU needed, sub-millisecond at scale.

### When BM25 wins
- Exact identifiers: error codes, SKUs, PINs, legal clause numbers  
- Rare technical terms that only appear in the target document  
- Queries where the user's vocabulary exactly matches the document's vocabulary

In [3]:
# --- Sample corpus ---------------------------------------------------------
DOCS = [
    {"id": 0, "text": "BM25 uses an inverted index for fast keyword retrieval."},
    {"id": 1, "text": "Dense vector search finds semantically similar documents using embeddings."},
    {"id": 2, "text": "Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row."},
    {"id": 3, "text": "Reciprocal Rank Fusion merges multiple ranked lists without score normalization."},
    {"id": 4, "text": "HNSW is an approximate nearest neighbour algorithm used in vector databases."},
    {"id": 5, "text": "Python database queries can be slow without proper indexing strategies."},
    {"id": 6, "text": "PostgreSQL optimization techniques include EXPLAIN ANALYZE and index hints."},
    {"id": 7, "text": "SKU-4821 is a premium standing desk with height-adjustable frame."},
    {"id": 8, "text": "Agentic RAG gives the language model a retrieval tool it can invoke iteratively."},
    {"id": 9, "text": "Hybrid search improves recall by 15-30% over single-method retrieval."},
]

# Tokenise (whitespace + lowercase — use a proper tokeniser in production)
tokenised_corpus = [doc["text"].lower().split() for doc in DOCS]

bm25 = BM25Okapi(tokenised_corpus)

def bm25_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, score) for doc_id, score in ranked[:top_k] if score > 0]

# Demo
query_kw = "ER_NO_DEFAULT_FOR_FIELD MySQL error"
results = bm25_search(query_kw)
print(f"BM25 results for: '{query_kw}'")
for doc_id, score in results:
    print(f"  [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")

BM25 results for: 'ER_NO_DEFAULT_FOR_FIELD MySQL error'
  [02] score=5.512  |  Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row.


---
## 3. kNN Dense Retrieval

### Theory

Dense retrieval encodes both queries and documents into a **continuous vector space** using a transformer encoder (e.g. `all-MiniLM-L6-v2`, `text-embedding-3-small`).

At index time:
$$\mathbf{e}_d = \text{Encoder}(d) \in \mathbb{R}^{n}$$

At query time, find the $k$ nearest document vectors:
$$\text{sim}(\mathbf{e}_q, \mathbf{e}_d) = \frac{\mathbf{e}_q \cdot \mathbf{e}_d}{\|\mathbf{e}_q\| \cdot \|\mathbf{e}_d\|}$$

**HNSW (Hierarchical Navigable Small World)** is the standard ANN index — sub-linear lookup in $O(\log N)$ rather than brute-force $O(N)$.

### When dense retrieval wins
- Synonym bridging: `"slow database queries"` → finds `"PostgreSQL optimization techniques"`  
- Paraphrase matching: `"how to fix import errors"` → finds `"resolving module not found exceptions"`  
- Conversational, conceptual, or multi-hop questions

In [4]:
# Build a FAISS flat index (exact cosine via inner-product on normalised vectors)
model = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = [doc["text"] for doc in DOCS]
corpus_embeddings = model.encode(corpus_texts, normalize_embeddings=True)  # L2-normalised → dot == cosine

dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product on unit vectors = cosine similarity
index.add(corpus_embeddings.astype("float32"))

print(f"FAISS index built: {index.ntotal} vectors of dim={dim}")


def dense_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx != -1]


# Demo — semantic query with no exact keyword overlap
query_sem = "database performance optimisation"
results = dense_search(query_sem)
print(f"\nDense results for: '{query_sem}'")
for doc_id, score in results:
    print(f"  [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20587.75it/s]


FAISS index built: 10 vectors of dim=384

Dense results for: 'database performance optimisation'
  [05] score=0.524  |  Python database queries can be slow without proper indexing strategies.
  [06] score=0.452  |  PostgreSQL optimization techniques include EXPLAIN ANALYZE and index hints.
  [00] score=0.286  |  BM25 uses an inverted index for fast keyword retrieval.
  [09] score=0.206  |  Hybrid search improves recall by 15-30% over single-method retrieval.
  [04] score=0.195  |  HNSW is an approximate nearest neighbour algorithm used in vector databases.


---
## 4. Fusion Strategies

### 4a. Reciprocal Rank Fusion (RRF)

RRF is **score-agnostic** — it ignores raw scores and only uses rank positions. This sidesteps the scale mismatch between BM25 (unbounded floats) and cosine similarity (−1 to 1).

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

- $k = 60$ is the constant from the original 2009 paper (empirically robust)
- A document ranked **#1** scores $\frac{1}{61} \approx 0.0164$
- A document appearing in **both** lists scores the sum of both contributions

**Properties:**
- Zero labelled data required
- Robust to score distribution mismatches
- Best starting point for any hybrid system

### 4b. Convex Combination (Weighted Score Fusion)

$$\text{score}(d) = \alpha \cdot \hat{s}_{\text{dense}}(d) + (1 - \alpha) \cdot \hat{s}_{\text{sparse}}(d)$$

Both scores are min-max normalised to $[0, 1]$ first. $\alpha$ controls the balance:

| $\alpha$ | Behaviour |
|----------|-----------|
| 1.0 | Pure dense (semantic) |
| 0.7 | Semantic-heavy — good for conversational queries |
| 0.5 | Balanced |
| 0.3 | Keyword-heavy — good for technical docs / error codes |
| 0.0 | Pure BM25 |

Requires 50–100 labelled query pairs to tune $\alpha$ effectively.

In [5]:
def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[int, float]], k: int = 60
) -> list[tuple[int, float]]:
    """Merge ranked lists using RRF. Each list is [(doc_id, score), ...]."""
    rrf_scores: dict[int, float] = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def minmax_normalize(scores: list[tuple[int, float]]) -> list[tuple[int, float]]:
    if not scores:
        return scores
    values = [s for _, s in scores]
    lo, hi = min(values), max(values)
    if hi == lo:
        return [(doc_id, 1.0) for doc_id, _ in scores]
    return [(doc_id, (s - lo) / (hi - lo)) for doc_id, s in scores]


def convex_combination(
    dense_results: list[tuple[int, float]],
    sparse_results: list[tuple[int, float]],
    alpha: float = 0.5,
) -> list[tuple[int, float]]:
    """Weighted score fusion. alpha=1 → pure dense; alpha=0 → pure BM25."""
    dense_norm = dict(minmax_normalize(dense_results))
    sparse_norm = dict(minmax_normalize(sparse_results))
    all_ids = set(dense_norm) | set(sparse_norm)
    combined = {
        doc_id: alpha * dense_norm.get(doc_id, 0.0) + (1 - alpha) * sparse_norm.get(doc_id, 0.0)
        for doc_id in all_ids
    }
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)


def hybrid_search(
    query: str, top_k: int = 5, fusion: str = "rrf", alpha: float = 0.5
) -> list[tuple[int, float]]:
    """Run BM25 + kNN then fuse. fusion='rrf' or 'convex'."""
    bm25_results = bm25_search(query, top_k=top_k * 2)
    dense_results = dense_search(query, top_k=top_k * 2)

    if fusion == "rrf":
        merged = reciprocal_rank_fusion(bm25_results, dense_results)
    else:
        merged = convex_combination(dense_results, bm25_results, alpha=alpha)

    return merged[:top_k]


def print_results(label: str, results: list[tuple[int, float]]) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print('─'*60)
    for doc_id, score in results:
        print(f"  [{doc_id:02d}] score={score:.4f}  |  {DOCS[doc_id]['text']}")

In [6]:
# ── Compare all three strategies on two contrasting queries ─────────────────

queries = {
    "exact identifier query": "ER_NO_DEFAULT_FOR_FIELD MySQL error",
    "semantic / paraphrase query": "how to speed up database queries",
}

for label, q in queries.items():
    print(f"\n{'═'*60}")
    print(f"  QUERY ({label}): {q}")
    print_results("BM25 only", bm25_search(q, top_k=3))
    print_results("Dense (kNN) only", dense_search(q, top_k=3))
    print_results("Hybrid — RRF (k=60)", hybrid_search(q, top_k=3, fusion="rrf"))
    print_results("Hybrid — Convex α=0.5", hybrid_search(q, top_k=3, fusion="convex", alpha=0.5))


════════════════════════════════════════════════════════════
  QUERY (exact identifier query): ER_NO_DEFAULT_FOR_FIELD MySQL error

────────────────────────────────────────────────────────────
  BM25 only
────────────────────────────────────────────────────────────
  [02] score=5.5124  |  Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row.

────────────────────────────────────────────────────────────
  Dense (kNN) only
────────────────────────────────────────────────────────────
  [02] score=0.8691  |  Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row.
  [05] score=0.0999  |  Python database queries can be slow without proper indexing strategies.
  [00] score=0.0573  |  BM25 uses an inverted index for fast keyword retrieval.

────────────────────────────────────────────────────────────
  Hybrid — RRF (k=60)
────────────────────────────────────────────────────────────
  [02] score=0.0328  |  Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL w

---
## 5. Agentic RAG with Hybrid Search

### From passive pipeline to active agent

**Passive RAG** — retrieval happens once, before the LLM:
```
User → retrieve(query) → [chunks] → LLM → answer
```

**Agentic RAG** — the LLM *decides* when and how to retrieve:
```
User → LLM (with tools) ─┬─ retrieve(query_1) → evaluate → ...
                          ├─ retrieve(query_2, alpha=0.2) → ...
                          └─ synthesise → answer
```

Benefits:
1. **Query rewriting** — the LLM rephrases the query before retrieval for better recall  
2. **Iterative retrieval** — if the first batch isn't enough, the agent retrieves again  
3. **Dynamic alpha selection** — the agent picks sparse/dense weight based on query type  
4. **Multi-hop reasoning** — retrieve → reason → retrieve again → answer  

### Implementation with Claude's tool use API

In [7]:
import anthropic

# ── Tool definition exposed to Claude ───────────────────────────────────────
HYBRID_SEARCH_TOOL = {
    "name": "hybrid_search",
    "description": (
        "Search the document corpus using hybrid BM25 + kNN retrieval. "
        "Use alpha close to 0 for exact keyword queries (error codes, IDs), "
        "alpha close to 1 for semantic/conceptual queries, and alpha=0.5 for mixed queries."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query. Rephrase for better retrieval if needed.",
            },
            "top_k": {
                "type": "integer",
                "description": "Number of documents to return (default 3, max 5).",
                "default": 3,
            },
            "alpha": {
                "type": "number",
                "description": "Weight for dense (semantic) retrieval. 0=BM25 only, 1=kNN only, 0.5=balanced.",
                "default": 0.5,
            },
        },
        "required": ["query"],
    },
}


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch tool calls from the LLM to our local hybrid search."""
    if tool_name == "hybrid_search":
        query = tool_input["query"]
        top_k = min(int(tool_input.get("top_k", 3)), 5)
        alpha = float(tool_input.get("alpha", 0.5))
        results = hybrid_search(query, top_k=top_k, fusion="convex", alpha=alpha)
        hits = [
            {"doc_id": doc_id, "score": round(score, 4), "text": DOCS[doc_id]["text"]}
            for doc_id, score in results
        ]
        return json.dumps({"results": hits}, indent=2)
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

In [8]:
def agentic_rag(user_question: str, max_iterations: int = 5, verbose: bool = True) -> str:
    """
    Agentic RAG loop.
    
    The agent autonomously decides:
    - What query to issue (query rewriting)
    - What alpha to use (sparse vs dense weight)
    - Whether to retrieve again (iterative retrieval)
    """
    client = anthropic.Anthropic()

    system_prompt = (
        "You are a helpful assistant with access to a document corpus via the hybrid_search tool. "
        "Before answering, always search for relevant documents. "
        "If the initial results are insufficient, search again with a different query or alpha value. "
        "Choose alpha carefully: use low alpha (≈0.1) for exact keyword lookups, "
        "high alpha (≈0.9) for conceptual questions, and 0.5 for general queries."
    )

    messages = [{"role": "user", "content": user_question}]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=system_prompt,
            tools=[HYBRID_SEARCH_TOOL],
            messages=messages,
        )

        if verbose:
            print(f"\n[Iteration {iteration}] stop_reason={response.stop_reason}")

        # Append assistant turn
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            # Extract final text response
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text response)"

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  Tool call: {block.name}({json.dumps(block.input)})")
                    result = execute_tool(block.name, block.input)
                    if verbose:
                        print(f"  Tool result: {result[:300]}...")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return "(max iterations reached)"

In [ ]:
# ── Run the agent on two questions ──────────────────────────────────────────
# API key is loaded from a .env file in the same directory as this notebook.
# Create a .env file with: ANTHROPIC_API_KEY=sk-ant-...

import os
from dotenv import load_dotenv

load_dotenv()  # loads .env from the current working directory

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠  ANTHROPIC_API_KEY not set — skipping live API calls.\n"
          "Create a .env file with ANTHROPIC_API_KEY=sk-ant-... and re-run.")
else:
    print("=" * 60)
    print("Question 1: Exact keyword lookup")
    print("=" * 60)
    answer1 = agentic_rag("What causes the ER_NO_DEFAULT_FOR_FIELD error?")
    print("\n── Final answer ──")
    print(answer1)

    print("\n" + "=" * 60)
    print("Question 2: Semantic / conceptual query")
    print("=" * 60)
    answer2 = agentic_rag("How can I make my database queries faster?")
    print("\n── Final answer ──")
    print(answer2)


Question 1: Exact keyword lookup

[Iteration 1] stop_reason=tool_use
  Tool call: hybrid_search({"query": "ER_NO_DEFAULT_FOR_FIELD error", "alpha": 0.1})
  Tool result: {
  "results": [
    {
      "doc_id": 2,
      "score": 1.0,
      "text": "Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row."
    },
    {
      "doc_id": 7,
      "score": 0.0123,
      "text": "SKU-4821 is a premium standing desk with height-adjustable frame."
    },
    {...
  Tool call: hybrid_search({"query": "no default value for field error cause", "alpha": 0.5})
  Tool result: {
  "results": [
    {
      "doc_id": 0,
      "score": 0.5289,
      "text": "BM25 uses an inverted index for fast keyword retrieval."
    },
    {
      "doc_id": 2,
      "score": 0.5,
      "text": "Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row."
    },
    {
      "do...

[Iteration 2] stop_reason=tool_use
  Tool call: hybrid_search({"query": "ER_NO_DEFAULT_FOR_FIELD MySQL insert mis

---
## 6. Evaluation Metrics

Three standard metrics for retrieval systems. All assume a **relevance judgement set** — a list of `(query, relevant_doc_ids)` pairs.

| Metric | What it measures | Formula |
|--------|-----------------|---------|
| **Hit Rate @K** | Does any relevant doc appear in top-K? | $\frac{1}{Q}\sum_q \mathbf{1}[\text{rel} \cap \text{top-K} \neq \emptyset]$ |
| **MRR** | How high is the *first* relevant doc? | $\frac{1}{Q}\sum_q \frac{1}{\text{rank}_{\text{first relevant}}}$ |
| **NDCG@K** | Graded relevance — rewards earlier hits more | $\frac{\text{DCG@K}}{\text{IDCG@K}}$ |

Higher is better for all three. NDCG@K is the most informative for production systems.

In [12]:
def hit_rate(results: list[tuple[int, float]], relevant_ids: set[int]) -> float:
    retrieved = {doc_id for doc_id, _ in results}
    return 1.0 if retrieved & relevant_ids else 0.0


def mrr(results: list[tuple[int, float]], relevant_ids: set[int]) -> float:
    for rank, (doc_id, _) in enumerate(results, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(results: list[tuple[int, float]], relevant_ids: set[int], k: int) -> float:
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, (doc_id, _) in enumerate(results[:k], start=1)
        if doc_id in relevant_ids
    )
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, min(len(relevant_ids), k) + 1))
    return dcg / idcg if idcg > 0 else 0.0


# ── Mini evaluation on our small corpus ─────────────────────────────────────
EVAL_SET = [
    {"query": "ER_NO_DEFAULT_FOR_FIELD MySQL error",    "relevant": {2}},
    {"query": "how to speed up database queries",       "relevant": {5, 6}},
    {"query": "approximate nearest neighbour vector DB","relevant": {4}},
    {"query": "fuse multiple ranked lists no scores",   "relevant": {3}},
    {"query": "agentic retrieval augmented generation", "relevant": {8, 9}},
]

strategies = {
    "BM25 only":      lambda q: bm25_search(q, top_k=5),
    "Dense only":     lambda q: dense_search(q, top_k=5),
    "Hybrid RRF":     lambda q: hybrid_search(q, top_k=5, fusion="rrf"),
    "Hybrid α=0.5":   lambda q: hybrid_search(q, top_k=5, fusion="convex", alpha=0.5),
}

print(f"{'Strategy':<20} {'Hit@3':>8} {'MRR':>8} {'NDCG@3':>8}")
print("─" * 48)

for name, retriever in strategies.items():
    hits, mrrs, ndcgs = [], [], []
    for item in EVAL_SET:
        res = retriever(item["query"])
        rel = item["relevant"]
        hits.append(hit_rate(res[:3], rel))
        mrrs.append(mrr(res, rel))
        ndcgs.append(ndcg_at_k(res, rel, k=3))
    print(f"{name:<20} {np.mean(hits):>8.3f} {np.mean(mrrs):>8.3f} {np.mean(ndcgs):>8.3f}")

Strategy                Hit@3      MRR   NDCG@3
────────────────────────────────────────────────
BM25 only               1.000    1.000    0.845
Dense only              1.000    1.000    1.000
Hybrid RRF              1.000    1.000    1.000
Hybrid α=0.5            1.000    1.000    1.000


---
## 7. GraphRAG — Graph-Augmented Retrieval

### Why GraphRAG?

Hybrid search (BM25 + kNN) retrieves **isolated chunks**. It still struggles when an answer requires **connecting information spread across multiple documents** — e.g.:

> *"What are the common themes across all our support tickets this quarter?"*  
> *"How are the entity A and entity B related?"*

GraphRAG (Microsoft Research, 2024) solves this by building a **knowledge graph** over the corpus, then using graph traversal and community summaries at query time.

### The Two-Phase Pipeline

```
── INDEXING ────────────────────────────────────────────────────────────
  1. Chunk corpus into TextUnits
  2. LLM extracts (Entity, Relationship, Claim) triples from each TextUnit
  3. Merge duplicate entities across chunks → knowledge graph nodes/edges
  4. Run Leiden community detection → hierarchical clusters
  5. LLM summarises each community → Community Summary stored per level

── QUERYING ────────────────────────────────────────────────────────────
  Local Search  → find seed entities → traverse neighbours → gather context
  Global Search → map-reduce over all Community Summaries → synthesise
```

### Local vs Global Search

| | Local Search | Global Search |
|---|---|---|
| **Entry point** | Specific entities (via vector/BM25) | All community summaries |
| **Traversal** | Expand to 1-2 hop neighbours | Map-reduce across hierarchy |
| **Best for** | Entity-specific questions | Holistic / thematic questions |
| **Cost** | Low (targeted) | High (full corpus scan) |

### Where it fits alongside Hybrid Search

```
Query → Hybrid Search (BM25 + kNN) → relevant chunks
      → GraphRAG Local Search       → entity + neighbour context
      → GraphRAG Global Search      → community themes
                    ↓ combine ↓
                LLM synthesis → answer
```

In [ ]:
%pip install -q networkx

### 7a. Build the Knowledge Graph (Indexing Phase)

In [ ]:
import networkx as nx

# ── Simulate the LLM extraction step ────────────────────────────────────────
# In production: for each TextUnit, call an LLM with a structured extraction
# prompt → parse (entity, relationship, entity) triples.
# Here we hand-code triples that mirror our DOCS corpus.

RAW_TRIPLES = [
    # (source_entity, relation, target_entity, source_doc_id)
    ("BM25",             "uses",              "inverted index",           0),
    ("BM25",             "extends",           "TF-IDF",                   0),
    ("kNN search",       "uses",              "dense embeddings",         1),
    ("kNN search",       "uses",              "HNSW index",               4),
    ("HNSW index",       "is type of",        "ANN algorithm",            4),
    ("dense embeddings", "produced by",       "transformer encoder",      1),
    ("hybrid search",    "combines",          "BM25",                     9),
    ("hybrid search",    "combines",          "kNN search",               9),
    ("hybrid search",    "uses fusion",       "Reciprocal Rank Fusion",   3),
    ("RRF",              "alias of",          "Reciprocal Rank Fusion",   3),
    ("Reciprocal Rank Fusion", "merges",      "ranked lists",             3),
    ("agentic RAG",      "wraps",             "hybrid search",            8),
    ("agentic RAG",      "uses",              "LLM tool use",             8),
    ("MySQL",            "raises error",      "ER_NO_DEFAULT_FOR_FIELD",  2),
    ("PostgreSQL",       "optimised via",     "EXPLAIN ANALYZE",          6),
    ("PostgreSQL",       "optimised via",     "index hints",              6),
    ("slow queries",     "fixed by",          "PostgreSQL",               5),
    ("slow queries",     "fixed by",          "index hints",              5),
    ("SKU-4821",         "is a",              "standing desk",            7),
]

# ── Build directed multigraph ────────────────────────────────────────────────
G = nx.DiGraph()

for src, rel, tgt, doc_id in RAW_TRIPLES:
    # Nodes carry a list of contributing doc IDs
    if not G.has_node(src):
        G.add_node(src, docs=set())
    if not G.has_node(tgt):
        G.add_node(tgt, docs=set())
    G.nodes[src]["docs"].add(doc_id)
    G.nodes[tgt]["docs"].add(doc_id)
    G.add_edge(src, tgt, relation=rel, doc_id=doc_id)

print(f"Knowledge graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("\nNodes:", sorted(G.nodes()))

### 7b. Community Detection (Leiden / Louvain)

In the real GraphRAG pipeline, **Leiden** clustering groups densely connected nodes into communities, then an LLM summarises each community. Here we use NetworkX's Louvain approximation on the undirected projection.

In [ ]:
from collections import defaultdict

# Louvain requires an undirected graph
G_undirected = G.to_undirected()

# networkx.community.louvain_communities available in networkx >= 3.0
communities = list(nx.community.louvain_communities(G_undirected, seed=42))

# Map node → community id
node_to_community: dict[str, int] = {}
for cid, community in enumerate(communities):
    for node in community:
        node_to_community[node] = cid

# Group by community
community_nodes: dict[int, list[str]] = defaultdict(list)
for node, cid in node_to_community.items():
    community_nodes[cid].append(node)

print("Communities detected:")
for cid, nodes in sorted(community_nodes.items()):
    print(f"  Community {cid}: {sorted(nodes)}")

# ── Simulate LLM-generated community summaries ──────────────────────────────
# In production: for each community, send the member entities + their edges
# to an LLM and ask for a paragraph summary.
COMMUNITY_SUMMARIES = {
    cid: f"[Community {cid}] Covers concepts: {', '.join(sorted(nodes)[:6])}{'...' if len(nodes) > 6 else ''}."
    for cid, nodes in community_nodes.items()
}

print("\nCommunity summaries (stub):")
for cid, summary in COMMUNITY_SUMMARIES.items():
    print(f"  {summary}")

### 7c. Local Search — Entity-Centric Graph Traversal

**Flow:** find seed entities matching the query → expand to 1-2 hop neighbours → gather the sub-graph text as context.

In [ ]:
def find_seed_entities(query: str, top_k: int = 3) -> list[str]:
    """Embed the query and find the most similar entity names via cosine similarity."""
    q_emb = model.encode([query], normalize_embeddings=True)
    entity_names = list(G.nodes())
    entity_embs = model.encode(entity_names, normalize_embeddings=True)
    sims = (entity_embs @ q_emb.T).flatten()
    ranked = sorted(zip(entity_names, sims), key=lambda x: x[1], reverse=True)
    return [name for name, _ in ranked[:top_k]]


def graph_local_search(query: str, hops: int = 2, seed_k: int = 3) -> dict:
    """
    GraphRAG local search.
    Returns seed entities, the sub-graph edges, and source doc chunks.
    """
    seeds = find_seed_entities(query, top_k=seed_k)

    # BFS expansion up to `hops` away from each seed
    visited_nodes: set[str] = set(seeds)
    frontier = set(seeds)
    for _ in range(hops):
        next_frontier: set[str] = set()
        for node in frontier:
            next_frontier.update(G.successors(node))
            next_frontier.update(G.predecessors(node))
        frontier = next_frontier - visited_nodes
        visited_nodes.update(frontier)

    # Collect edges within the sub-graph
    subgraph_edges = [
        (u, data["relation"], v, data["doc_id"])
        for u, v, data in G.edges(data=True)
        if u in visited_nodes and v in visited_nodes
    ]

    # Collect source doc chunks
    doc_ids = {data["doc_id"] for _, _, data in G.edges(data=True) if data["doc_id"] in range(len(DOCS))}
    source_chunks = [DOCS[did]["text"] for did in sorted(doc_ids) if did < len(DOCS)]

    return {
        "seeds": seeds,
        "subgraph_nodes": sorted(visited_nodes),
        "subgraph_edges": subgraph_edges,
        "source_chunks": source_chunks,
    }


# Demo
result = graph_local_search("how does hybrid search work?")
print("Seeds:", result["seeds"])
print("\nSub-graph edges:")
for u, rel, v, did in result["subgraph_edges"]:
    print(f"  ({u}) --[{rel}]--> ({v})  [doc {did}]")
print("\nSource chunks:")
for chunk in result["source_chunks"]:
    print(f"  · {chunk}")

### 7d. Global Search — Map-Reduce over Community Summaries

**Flow:** retrieve the most relevant community summaries (map step) → LLM synthesises a final answer across all of them (reduce step). Used for holistic, thematic questions that span the whole corpus.

In [ ]:
# Embed community summaries once at index time
community_ids = sorted(COMMUNITY_SUMMARIES.keys())
summary_texts = [COMMUNITY_SUMMARIES[cid] for cid in community_ids]
summary_embeddings = model.encode(summary_texts, normalize_embeddings=True)


def graph_global_search(query: str, top_communities: int = 3) -> dict:
    """
    GraphRAG global search (map step).
    Ranks community summaries by semantic relevance to the query,
    returns the top-K for the LLM reduce step.
    """
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = (summary_embeddings @ q_emb.T).flatten()
    ranked = sorted(zip(community_ids, sims), key=lambda x: x[1], reverse=True)
    top = ranked[:top_communities]
    return {
        "query": query,
        "relevant_communities": [
            {"community_id": cid, "score": float(score), "summary": COMMUNITY_SUMMARIES[cid]}
            for cid, score in top
        ],
    }


result_global = graph_global_search("What are the main retrieval and indexing concepts in the corpus?")
print(f"Global search: '{result_global['query']}'")
for item in result_global["relevant_communities"]:
    print(f"\n  Community {item['community_id']} (score={item['score']:.3f})")
    print(f"  {item['summary']}")

### 7e. Combined Retrieval — Hybrid Search + GraphRAG

In a production agentic system, expose **three tools** to the LLM and let it pick the right one per query turn:

| Tool | Input | Best for |
|------|-------|----------|
| `hybrid_search` | query string, alpha | Chunk-level fact lookup |
| `graph_local_search` | query, hops | Entity relationships, multi-hop reasoning |
| `graph_global_search` | query | Thematic / holistic questions |

In [ ]:
GRAPH_LOCAL_TOOL = {
    "name": "graph_local_search",
    "description": (
        "Search the knowledge graph starting from entities related to the query, "
        "then expand to neighbouring nodes (multi-hop). Use for questions about "
        "relationships, causes, or specific entities and their connections."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The entity or concept to find."},
            "hops": {
                "type": "integer",
                "description": "Graph traversal depth (1=direct neighbours, 2=2-hop). Default 2.",
                "default": 2,
            },
        },
        "required": ["query"],
    },
}

GRAPH_GLOBAL_TOOL = {
    "name": "graph_global_search",
    "description": (
        "Search across all community summaries of the knowledge graph. "
        "Use for broad, thematic, or holistic questions that span the whole corpus "
        "(e.g. 'what are the main topics?', 'summarise the key concepts')."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The thematic question."},
            "top_communities": {
                "type": "integer",
                "description": "How many community summaries to return (default 3).",
                "default": 3,
            },
        },
        "required": ["query"],
    },
}


def execute_tool_extended(tool_name: str, tool_input: dict) -> str:
    """Extended dispatcher that handles hybrid + graph tools."""
    if tool_name == "hybrid_search":
        return execute_tool(tool_name, tool_input)  # reuse from section 5

    if tool_name == "graph_local_search":
        query = tool_input["query"]
        hops = int(tool_input.get("hops", 2))
        result = graph_local_search(query, hops=hops)
        return json.dumps(result, indent=2)

    if tool_name == "graph_global_search":
        query = tool_input["query"]
        top_k = int(tool_input.get("top_communities", 3))
        result = graph_global_search(query, top_communities=top_k)
        return json.dumps(result, indent=2)

    return json.dumps({"error": f"Unknown tool: {tool_name}"})


def agentic_rag_with_graph(user_question: str, max_iterations: int = 6, verbose: bool = True) -> str:
    """Agentic RAG with three retrieval tools: hybrid, local graph, global graph."""
    client = anthropic.Anthropic()

    system_prompt = (
        "You are a research assistant with three retrieval tools:\n"
        "1. hybrid_search — best for specific facts, error codes, or keyword queries\n"
        "2. graph_local_search — best for relationship and multi-hop entity questions\n"
        "3. graph_global_search — best for broad thematic or holistic questions\n\n"
        "Always retrieve before answering. Choose the tool that best fits the question type. "
        "You may use multiple tools in sequence if needed."
    )

    messages = [{"role": "user", "content": user_question}]
    all_tools = [HYBRID_SEARCH_TOOL, GRAPH_LOCAL_TOOL, GRAPH_GLOBAL_TOOL]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=system_prompt,
            tools=all_tools,
            messages=messages,
        )

        if verbose:
            print(f"\n[Iteration {iteration}] stop_reason={response.stop_reason}")

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text response)"

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  Tool: {block.name}({json.dumps(block.input)})")
                    result = execute_tool_extended(block.name, block.input)
                    if verbose:
                        print(f"  Result: {result[:300]}...")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return "(max iterations reached)"

In [ ]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠  ANTHROPIC_API_KEY not set — skipping live API calls.")
else:
    print("=" * 60)
    print("Question: relationship / multi-hop → expect graph_local_search")
    print("=" * 60)
    a = agentic_rag_with_graph("What algorithms and data structures does hybrid search rely on?")
    print("\n── Final answer ──\n", a)

    print("\n" + "=" * 60)
    print("Question: thematic → expect graph_global_search")
    print("=" * 60)
    b = agentic_rag_with_graph("Give me a high-level overview of all the main topics in the corpus.")
    print("\n── Final answer ──\n", b)

---
## 8. Decision Guide — Which Strategy to Use?

```
                          What kind of query?
                                 │
         ┌───────────────────────┼────────────────────────┐
         ▼                       ▼                        ▼
   Exact identifiers       Mixed / general          Conceptual /
   (error codes,           queries                  semantic
    SKUs, IDs)                   │                  (synonyms,
         │                       │                   paraphrase)
         ▼                       ▼                        ▼
  Convex α ≈ 0.1           RRF (k=60)             Convex α ≈ 0.9
  or BM25 only         or Convex α=0.5            or Dense only

         ┌───────────────────────┼────────────────────────┐
         ▼                       ▼                        ▼
   Multi-hop /            Thematic /               Standard
   relationship           holistic                 fact lookup
         │                       │                        │
         ▼                       ▼                        ▼
  Graph Local             Graph Global             Hybrid Search
  Search                  Search                   (BM25 + kNN)
```

| Situation | Tool | Setting |
|-----------|------|---------|
| Exact identifiers, no labelled data | Hybrid RRF | k=60 |
| 50+ labelled pairs available | Hybrid Convex | Tune α |
| Technical docs, error codes | Hybrid Convex | α ≈ 0.2–0.3 |
| Conversational / support KB | Hybrid Convex | α ≈ 0.7–0.8 |
| Entity relationships / multi-hop | GraphRAG Local | hops=2 |
| "What are all the themes?" | GraphRAG Global | top_communities=3–5 |
| Agentic system | All three as tools | LLM picks dynamically |

> **Empirical gains on BEIR benchmark:**  
> Hybrid vs dense-only: **+26–31% NDCG** on high vocabulary-mismatch domains  
> Hybrid vs BM25-only: **+18.5% MRR** on semantic / paraphrase-heavy domains  
> GraphRAG vs baseline RAG: significant gains on **cross-document reasoning** and **thematic summarisation**

---
## 9. Production Checklist

### Hybrid Search
```
□  Tokenisation        Use a proper tokeniser (NLTK, spaCy) — not just whitespace split
□  Stopword removal    Strip "the/a/is" before BM25 indexing
□  Chunking strategy   500–1000 token chunks with 10–20% overlap
□  Embedding model     Match embedding model to your domain (fine-tune if needed)
□  Index type          FAISS HNSW (local) / Qdrant / Weaviate / Elasticsearch (production)
□  Reranking           Add a cross-encoder reranker (ms-marco-MiniLM) on top-20 hits
□  Alpha tuning        Build ≥100 labelled query pairs; grid-search α before going live
□  Caching             Cache doc embeddings at index time; cache query embeddings for repeats
```

### GraphRAG
```
□  Extraction prompt   Invest in a good entity/relation extraction prompt — quality drives graph quality
□  Entity resolution   Merge duplicate entity names (e.g. "PostgreSQL" vs "Postgres") before building G
□  Community levels    Build multiple hierarchy levels; use level 0 (fine) for local, level 2+ for global
□  Summary quality     LLM-generated community summaries are the "index" — review a sample before shipping
□  Graph DB            For large corpora use Neo4j / Kuzu instead of NetworkX
□  LazyGraphRAG        For cost-sensitive use cases: skip pre-summarising; summarise on-demand at query time
```

### Agentic System
```
□  Tool descriptions   Clear, contrasting descriptions so the LLM picks the right tool
□  Monitoring          Log which tool the agent picks per query — tool choice distribution is a health signal
□  Max iterations      Cap agentic loops (5–8 iterations) to prevent runaway costs
□  Fallback            If graph search returns empty, fall back to hybrid search automatically
```

---

## Sources

- [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)
- [From Local to Global: A GraphRAG Approach to Query-Focused Summarization — arXiv 2404.16130](https://arxiv.org/abs/2404.16130)
- [Microsoft GraphRAG Documentation](https://microsoft.github.io/graphrag/)
- [Hybrid Search for RAG: BM25, SPLADE, and Vector Search Combined — PremAI Blog](https://blog.premai.io/hybrid-search-for-rag-bm25-splade-and-vector-search-combined/)
- [Hybrid Search Explained — Weaviate](https://weaviate.io/blog/hybrid-search-explained)
- [Introducing Reciprocal Rank Fusion — OpenSearch](https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/)
- [GraphRAG Concepts — graphrag.com](https://graphrag.com/concepts/intro-to-graphrag/)